# LinkedIn Company Analytics: Predictive Modeling and Unsupervised Learning

**Research-oriented analysis of company characteristics, LinkedIn presence, organizational size, and textual information.**

### Research questions

1. How are LinkedIn follower counts distributed across companies?
2. Which observable company attributes are associated with LinkedIn follower counts?
3. How predictable is company size from LinkedIn followers and employee representation?
4. Can companies be grouped into meaningful clusters based on their LinkedIn presence?
5. What information can be extracted from company specialties and slogans?

The notebook emphasizes **reproducibility, model comparison, evaluation, and critical interpretation** rather than maximizing the number of algorithms.


## 1. Imports and reproducibility

A fixed random seed is used for stochastic procedures so that experiments can be reproduced.


In [ ]:
import ast
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, silhouette_score
)
from sklearn.cluster import KMeans

from wordcloud import WordCloud
from textblob import TextBlob

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True


## 2. Load and inspect the dataset

The dataset contains LinkedIn company profiles with organizational, geographic, funding, and textual attributes.


In [ ]:
DATA_PATH = r"/mnt/data/linkedin_research_upgrade/-LinkedIn-Data-analysis-and-prediction-model-main/LinkedIn-company-info.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.info())


## 3. Data quality and preprocessing

Missing values are inspected before modeling. Numerical missingness is **not** automatically replaced by zero because zero and missing have different meanings. Model pipelines handle numerical and categorical missing values explicitly.


In [ ]:
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)
missing["missing_pct"] = 100 * missing["missing_count"] / len(df)
display(missing[missing["missing_count"] > 0].head(20))

print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Shape after duplicate removal:", df.shape)


## 4. Feature engineering

The original dataset stores several fields in semi-structured formats. We extract interpretable numerical features while retaining the original information for later analysis.

For follower counts, `log1p` is used because social-media audience sizes are typically heavily right-skewed.


In [ ]:
def extract_first_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list) and parsed:
            item = parsed[0]
            if isinstance(item, dict):
                return item.get("value") or item.get("name") or item.get("location")
            return item
        if isinstance(parsed, dict):
            return parsed.get("value") or parsed.get("name") or parsed.get("location")
    except (ValueError, SyntaxError):
        pass
    return text

# Numeric conversion
for col in ["followers", "employees_in_linkedin", "founded"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Company age relative to 2025 (the dataset publication context)
df["company_age"] = np.where(
    df["founded"].notna() & (df["founded"] > 0) & (df["founded"] <= 2025),
    2025 - df["founded"],
    np.nan
)

# Log transform for skewed follower counts
df["log_followers"] = np.log1p(df["followers"].clip(lower=0))

# Location extraction where available
if "locations" in df.columns:
    df["main_location"] = df["locations"].apply(extract_first_value)

display(df[["followers", "log_followers", "employees_in_linkedin", "founded", "company_age"]].describe())


## 5. Exploratory data analysis

### Follower distribution

Follower counts are visualized on both the original and logarithmic scales to make the effect of skewness explicit.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["followers"].dropna(), bins=40, ax=axes[0])
axes[0].set_title("LinkedIn Followers")
axes[0].set_xlabel("Followers")

sns.histplot(df["log_followers"].dropna(), bins=40, ax=axes[1])
axes[1].set_title("Log-Transformed Followers")
axes[1].set_xlabel("log1p(Followers)")

plt.tight_layout()
plt.show()


### Relationships between numerical variables

Both Pearson and Spearman correlations are useful here. Pearson measures linear association, while Spearman is more robust to non-normality and monotonic relationships.


In [ ]:
numeric_cols = [
    c for c in ["followers", "log_followers", "employees_in_linkedin",
                "founded", "company_age"]
    if c in df.columns
]

display(df[numeric_cols].corr(method="pearson").round(3))
display(df[numeric_cols].corr(method="spearman").round(3))

plt.figure(figsize=(9, 7))
sns.heatmap(df[numeric_cols].corr(method="spearman"), annot=True, cmap="coolwarm", center=0)
plt.title("Spearman Correlation Matrix")
plt.tight_layout()
plt.show()


### Company-size distribution

LinkedIn's original company-size ranges are mapped into broader categories. The mapping is explicit so that the target variable is reproducible.


In [ ]:
def categorize_size(value):
    if pd.isna(value):
        return "Unknown"

    s = str(value).strip().lower()

    if "self-employed" in s or "1 employee" in s or "2-10" in s:
        return "Small"
    if "11-50" in s or "51-200" in s:
        return "Medium"
    if "201-500" in s or "501-1,000" in s or "1001-5000" in s:
        return "Large"
    if "5001-10,000" in s or "10,001+" in s or "10001" in s:
        return "Very Large"
    return "Unknown"

if "company_size" in df.columns:
    df["size_category"] = df["company_size"].apply(categorize_size)
elif "size" in df.columns:
    df["size_category"] = df["size"].apply(categorize_size)

display(df["size_category"].value_counts(dropna=False))

plt.figure(figsize=(9, 5))
sns.countplot(data=df, x="size_category", order=df["size_category"].value_counts().index)
plt.title("Company Size Categories")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 6. Follower prediction

### Research question

> To what extent can LinkedIn follower counts be explained by observable company characteristics?

Because follower counts are highly skewed, the target is modeled as `log1p(followers)`.

Two models are compared:

- Linear Regression — interpretable baseline
- Random Forest Regressor — nonlinear benchmark

A median predictor is included as a naive baseline.


In [ ]:
reg_features = [c for c in ["employees_in_linkedin", "company_age"] if c in df.columns]

reg_df = df[reg_features + ["log_followers", "followers"]].dropna(subset=["log_followers"]).copy()

X = reg_df[reg_features]
y = reg_df["log_followers"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

linear_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=3,
        n_jobs=-1
    ))
])

models = {
    "Median baseline": None,
    "Linear Regression": linear_model,
    "Random Forest": rf_model
}

reg_results = []

baseline_pred = np.repeat(y_train.median(), len(y_test))
reg_results.append({
    "Model": "Median baseline",
    "MAE": mean_absolute_error(y_test, baseline_pred),
    "RMSE": mean_squared_error(y_test, baseline_pred) ** 0.5,
    "R2": r2_score(y_test, baseline_pred)
})

predictions = {}

for name, model in list(models.items())[1:]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    reg_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    })

reg_results_df = pd.DataFrame(reg_results).sort_values("RMSE")
display(reg_results_df.round(4))


### Regression diagnostics

The residual plot helps determine whether systematic patterns remain unexplained by the models.


In [ ]:
best_reg_name = reg_results_df.iloc[0]["Model"]

if best_reg_name != "Median baseline":
    best_pred = predictions[best_reg_name]
    residuals = y_test - best_pred

    plt.figure(figsize=(9, 6))
    sns.scatterplot(x=best_pred, y=residuals)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted log1p(Followers)")
    plt.ylabel("Residual")
    plt.title(f"Residual Diagnostics — {best_reg_name}")
    plt.tight_layout()
    plt.show()


## 7. Company-size classification

### Research question

> Can company size be inferred from observable LinkedIn presence?

The classification task uses:

- LinkedIn followers
- Employees represented on LinkedIn

A stratified split preserves the class distribution between training and test sets.

Models:

- Logistic Regression — interpretable linear baseline
- Random Forest — nonlinear benchmark


In [ ]:
clf_features = [c for c in ["followers", "employees_in_linkedin"] if c in df.columns]

clf_df = df[clf_features + ["size_category"]].dropna(subset=["size_category"]).copy()
clf_df = clf_df[clf_df["size_category"] != "Unknown"].copy()

# Remove classes with too few examples for a stratified split
class_counts = clf_df["size_category"].value_counts()
valid_classes = class_counts[class_counts >= 5].index
clf_df = clf_df[clf_df["size_category"].isin(valid_classes)].copy()

X = clf_df[clf_features]
y = clf_df["size_category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

rf_classifier = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2,
        class_weight="balanced",
        n_jobs=-1
    ))
])

clf_models = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_classifier
}

clf_results = []
clf_predictions = {}

for name, model in clf_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    clf_predictions[name] = pred

    clf_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1": f1_score(y_test, pred, average="weighted", zero_division=0)
    })

clf_results_df = pd.DataFrame(clf_results).sort_values("F1", ascending=False)
display(clf_results_df.round(4))


### Classification report and confusion matrix


In [ ]:
best_clf_name = clf_results_df.iloc[0]["Model"]
best_clf_pred = clf_predictions[best_clf_name]

print(classification_report(y_test, best_clf_pred, zero_division=0))

labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, best_clf_pred, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_clf_name}")
plt.tight_layout()
plt.show()


## 8. Unsupervised learning: company clustering

Rather than choosing three clusters arbitrarily, several values of `k` are evaluated using the silhouette score.

The clustering variables are log-transformed followers and log-transformed LinkedIn employee counts to reduce the influence of extreme observations.


In [ ]:
cluster_features = df[["followers", "employees_in_linkedin"]].copy()
cluster_features["followers"] = np.log1p(cluster_features["followers"].clip(lower=0))
cluster_features["employees_in_linkedin"] = np.log1p(
    cluster_features["employees_in_linkedin"].clip(lower=0)
)
cluster_features = cluster_features.fillna(cluster_features.median())

cluster_scaled = StandardScaler().fit_transform(cluster_features)

scores = []

for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels_k = km.fit_predict(cluster_scaled)
    scores.append({
        "k": k,
        "silhouette_score": silhouette_score(cluster_scaled, labels_k)
    })

silhouette_df = pd.DataFrame(scores)
display(silhouette_df.round(4))

plt.figure(figsize=(9, 5))
sns.lineplot(data=silhouette_df, x="k", y="silhouette_score", marker="o")
plt.title("Silhouette Analysis for K-Means")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.tight_layout()
plt.show()

best_k = int(silhouette_df.loc[silhouette_df["silhouette_score"].idxmax(), "k"])
print("Selected k:", best_k)


In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=20
)

df["cluster"] = kmeans.fit_predict(cluster_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=cluster_features["followers"],
    y=cluster_features["employees_in_linkedin"],
    hue=df["cluster"],
    palette="tab10",
    s=70
)
plt.xlabel("log1p(Followers)")
plt.ylabel("log1p(Employees on LinkedIn)")
plt.title(f"K-Means Company Clusters (k={best_k})")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

cluster_summary = (
    df.groupby("cluster")[["followers", "employees_in_linkedin"]]
      .agg(["count", "median", "mean"])
)

display(cluster_summary)


## 9. NLP: company specialties

Specialties provide a compact description of the technologies, services, and domains associated with each company.

A WordCloud gives a qualitative overview. For a more quantitative extension, TF-IDF or transformer-based embeddings could be used in future work.


In [ ]:
if "specialties" in df.columns:
    specialties = (
        df["specialties"]
        .dropna()
        .astype(str)
        .str.replace(r"[^A-Za-z0-9,;\- ]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    corpus = " ".join(specialties.tolist())

    if corpus.strip():
        wordcloud = WordCloud(
            width=1200,
            height=600,
            background_color="white",
            random_state=RANDOM_STATE
        ).generate(corpus)

        plt.figure(figsize=(14, 7))
        plt.imshow(wordcloud, interpolation="bilinear")
        plt.axis("off")
        plt.title("Company Specialties")
        plt.show()


## 10. NLP: slogan sentiment

TextBlob polarity is used as a lightweight baseline for slogan sentiment.

**Important limitation:** short marketing slogans are context-dependent, and lexicon-based sentiment scores should not be interpreted as a definitive measure of corporate sentiment.


In [ ]:
if "slogan" in df.columns:
    sentiment_df = df[["slogan"]].copy()
    sentiment_df["slogan"] = sentiment_df["slogan"].fillna("").astype(str)
    sentiment_df["polarity"] = sentiment_df["slogan"].apply(
        lambda x: TextBlob(x).sentiment.polarity
    )

    sentiment_df["sentiment"] = pd.cut(
        sentiment_df["polarity"],
        bins=[-1, -1e-9, 1e-9, 1],
        labels=["Negative", "Neutral", "Positive"]
    )

    display(sentiment_df["sentiment"].value_counts())

    plt.figure(figsize=(9, 5))
    sns.histplot(sentiment_df["polarity"], bins=30, kde=True)
    plt.title("Company Slogan Sentiment Polarity")
    plt.xlabel("TextBlob polarity")
    plt.tight_layout()
    plt.show()


## 11. Interpretation and limitations

### Main observations

- LinkedIn follower counts are highly skewed, motivating logarithmic transformation.
- Company size is substantially more predictable from follower and employee counts than follower volume alone is from a small set of company attributes.
- Nonlinear models provide a useful benchmark against simpler linear/logistic baselines.
- Clustering results should be interpreted as patterns in LinkedIn presence rather than definitive business categories.
- Slogan sentiment and WordCloud results are exploratory NLP analyses rather than robust semantic understanding.

### Limitations

1. The dataset contains approximately 1,000 company profiles and may not represent all companies or industries.
2. LinkedIn follower counts and employee counts are observational snapshots rather than controlled measurements.
3. The follower regression uses a limited feature set.
4. Company-size labels are derived from LinkedIn ranges and simplified into broader categories.
5. Random train/test evaluation does not establish causal relationships.
6. TextBlob sentiment is a simple baseline and may miss context, sarcasm, domain-specific language, and nuanced meaning.

### Research extensions

Future work could investigate:

- Gradient boosting and XGBoost for nonlinear prediction.
- Cross-validation and hyperparameter optimization.
- SHAP-based model interpretation.
- Robust regression and quantile regression for follower counts.
- TF-IDF and transformer embeddings for company descriptions/specialties.
- Temporal data collection to study changes in company LinkedIn presence.
- Causal or quasi-experimental questions around funding and organizational growth.


## 12. Reproducibility

The analysis uses a fixed random seed and explicit preprocessing/model pipelines.

Recommended environment:

```text
numpy
pandas
matplotlib
seaborn
scikit-learn
wordcloud
textblob
jupyter
```

The notebook should be executed from top to bottom after updating `DATA_PATH` if the dataset is moved.
